<a href="https://colab.research.google.com/github/mariacastrom00/AI_exercises_MJCM/blob/main/Exercise_2_regression_model/Assignment_1_Regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Assignment 1: Demand prediction of bike sharing system**

Bike sharing services are nowadays spread all over the world. They are beneficial for cities as they offer a climate neutral and healthy way of transportation. Moreover, similar to e-scooters, for the companies and transportation researchers they generate valuable data. This includes the trip route, trip origins and destinations, duration, and much more.

In this assignment we use a public bike sharing dataset from Washington, D.C., USA. published by Fanaee-T, H. (2013). Bike Sharing [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5W894.

## Your Task

Your task in this assignment is:

1) To predict the demand of hourly rentals (**target-column: cnt**) using the [XGBoost](https://xgboost.readthedocs.io/en/stable/) and linear regression model. In this notebook we have provided you with the basic structure how to approach the problem. Feel free to reuse and improve parts of the exercise material.

2) After you have created the model please upload your code to Github and post the link to the notebook in the corresponding forum for the assignment with a short text describing how you have solved the task.

## Load data

You can also find more information about the data and the columns at: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/zhenliangma/Applied-AI-in-Transportation/master/Exercise_2_regression_model/Exercise2BikeSharing.csv'
df = pd.read_csv(url)

#df = pd.read_csv('Exercise2BikeSharing.csv')
df.head(10)

In [ ]:
df.shape

In [ ]:
df.info()
# atemp: How many degrees "you feel" it is
# casual: Count of casual users
# cnt: Total amount of bycicles

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

sns.histplot(x=df['cnt'])

## Train / Test split
- Target: `cnt` (count of total rental bikes including both casual and registered)
- Predictors: weather (`temp`, `atemp`, `hum`, `windspeed`, `weathersit`), calendar (`hr`, `weekday`, `workingday`, `holiday`, `season`), and `yr`.
- We keep it simple; you can expand features (e.g., interactions) later.

In [ ]:
# The format of dteday is object, information can be retrieved with other information

df.drop(columns=['dteday'], inplace=True)

In [ ]:
# Checking linear correlation between parameters and our given target "cnt"
# Initial way to choose which parameters to exclude

corr_matrix = df.corr()
corr_matrix['cnt'].sort_values(ascending=False)

In [ ]:
# Separating the data, target y, predictor variables X

from sklearn.model_selection import train_test_split
target = 'cnt'

X_original = df.copy()
y = df[target].astype(float) # Converts all data to float

X_original.drop(columns=[target], inplace=True)


## Data Processing / Feature Engineering

Before training the model think about data processing and feature engineering steps if applicable, such as:
- adding/removal of features
- normalization
- one hot encoding
- ...


In [ ]:
# CHOOSE THE FIRST!!!!

# What: instant, casual, registered, atemp. Why: atemp felt like a combination of windspeed+temp+wheathersit (redundant to have with temp),
# instant felt unncessesary (observation identifier), casual and registered was the sum of cnt so it felt like overflowing of information (prevent target leakage).
# Score: 0.7697409247091815
# X_original.drop(columns=['instant', 'casual', 'registered', 'atemp'], inplace=True)

# What: workingday, weekday, holiday. Why: When doing the correlation matrix, these had the lowest values
# Score: 0.9999930860825843
# Feels bad? Too god to be true...
# X_original.drop(columns=['workingday', 'weekday', 'holiday', 'atemp'], inplace=True)

# What: workingday, weekday, holiday. Why: When doing the correlation matrix, these had the lowest values
# Score: 0.5258824754737466
X_original.drop(columns=['workingday', 'weekday', 'holiday', 'atemp', 'casual', 'registered', 'instant'], inplace=True)


In [ ]:
# AI generated!
# One hot enconding - Improved the model very much

# Drop 1 use this
#X_encoded = pd.get_dummies(
#    X_original, columns=['season',
#                         'mnth',
#                         'weekday',
#                         'weathersit',
#                         'hr'],
#    drop_first=True)

# Drop 2 use this:
#X_encoded = pd.get_dummies(
#    X_original,
#    columns=[
#        'season',
#        'mnth',
#        'weathersit'
#    ], drop_first=True)

# Drop 3 use this:
X_encoded = pd.get_dummies(
    X_original, columns=['season',
                         'weathersit',
                         'hr'],
    drop_first=True)

In [ ]:
# How are all the predictor variables related to each other?
x = df.drop(['cnt'], axis=1)
y = df['cnt']

sns.pairplot(x)


# How are my selected predictors related to each other and to cnt?
# plot_df = X.copy()
# plot_df['cnt'] = y

# sns.pairplot(plot_df)

In [ ]:
# Train/test split
from sklearn.model_selection import train_test_split

X_train_orig, X_test_orig, y_train, y_test = train_test_split(
    X_original,
    y,
    test_size=0.2,
    random_state=42
)

X_train_enc, X_test_enc, _, _ = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Normalizing


In [ ]:
from sklearn.preprocessing import StandardScaler

# Normalize the features
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled = scaler.transform(X_test_enc)

## Hyperparameter optimization
Hyperparameter tuning helps find the best set of hyperparameters for your model. Use Grid Search or Random Search with Cross Validation from scikit-learn to search for the best combination of parameters.

In [ ]:
# Grid Search + Cross Validation

# Grid Search: Test different combinations of hyperparameters, search for
             # the best combination for highest model performance.

# Cross Validation: How well the model generalized, splits training data into
                    # many parts, traines some and validates the other ones.

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

# Define a parameter grid for hyperparameter tuning
param_grid = {
    'kernel': ['linear', 'rbf'],
    'C': [ 1, 10],
    'epsilon': [ 1, 10]
}

# Create the GridSearchCV object
grid_search = GridSearchCV(SVR(), param_grid, cv=5, verbose=2)

# Fit the grid search to the scaled training data
grid_search.fit(X_train_scaled, y_train)

# Get the best parameters
best_params = grid_search.best_params_

print("Best Parameters:", best_params)
print("Best Score:", grid_search.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END ......................C=1, epsilon=1, kernel=linear; total time=   6.9s
[CV] END ......................C=1, epsilon=1, kernel=linear; total time=   7.3s
[CV] END ......................C=1, epsilon=1, kernel=linear; total time=   7.1s
[CV] END ......................C=1, epsilon=1, kernel=linear; total time=   7.0s
[CV] END ......................C=1, epsilon=1, kernel=linear; total time=   7.3s
[CV] END .........................C=1, epsilon=1, kernel=rbf; total time=  11.5s
[CV] END .........................C=1, epsilon=1, kernel=rbf; total time=  10.6s
[CV] END .........................C=1, epsilon=1, kernel=rbf; total time=  10.5s
[CV] END .........................C=1, epsilon=1, kernel=rbf; total time=  10.7s
[CV] END .........................C=1, epsilon=1, kernel=rbf; total time=  10.6s
[CV] END .....................C=1, epsilon=10, kernel=linear; total time=   5.9s
[CV] END .....................C=1, epsilon=10, ke

## Train the XGBoost and Linear regression models
Now, create an XGBoost and linear regression model using the best parameters and train them using the training data.

In [ ]:
# XGBoost

from xgboost import XGBRegressor

# Create XGBoost model
xgb_model = XGBRegressor(random_state=42)

# Fit the model to the training data
# xgb_model.fit(X_train, y_train)
xgb_model.fit(X_train_orig, y_train)


# Predict the test data wit hthe fitted model
# y_pred_xgb = xgb_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test_orig)


In [ ]:
# Linear regression model
from sklearn.linear_model import LinearRegression

# Create a Linear Regression model
linear_model = LinearRegression()

# Fit the model to the training data
# model.fit(X_train, y_train)
linear_model.fit(X_train_enc, y_train)


# Predict the test data with the fitted model
# y_pred = model.predict(X_test)
y_pred_linear = linear_model.predict(X_test_enc)

## Make predictions
Use the trained models to make predictions on the test data and compare the performance of the XGBoost and linear regression models using metrics like Mean Squared Error (MSE) and R-squared.

In [ ]:
# Evaluate XGBoost model
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)

print(f"Mean Absolute Error XGBoost: {mae_xgb}")
print(f"Mean Squared Error XGBoost: {mse_xgb}")
print(f"RMSE XGBoost: {rmse_xgb}")
print(f"R-squared XGBoost: {r2_xgb}")

In [ ]:
# Evaluating the model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_linear = mean_absolute_error(y_test, y_pred_linear)
mse_linear = mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)
rmse_linear = np.sqrt(mse_linear)


print(f"Mean Absolute Error Linear Model: {mae_linear}")
print(f"Mean Squared Error Linear Model: {mse_linear}")
print(f"RMSE Linear: {rmse_linear}")
print(f"R-squared Linear Model: {r2_linear}")

## Visualize the predictions and compare the mdoels
Create a "Actual vs. Predicted Values" graph to give a visual inspection of the prediction quality.

In [ ]:
import matplotlib.pyplot as plt
# y_test contains the actual target values for the test dataset
# y_pred contains the predicted values for the test dataset

# Create a scatter plot to visualize the relationship
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_xgb, alpha=0.5)  # Plot actual vs. predicted values

# Add labels and title
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs. Predicted Values, XGBoost")

# Add a diagonal line for reference (perfect predictions)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], linestyle='--', color='red', lw=2)

# Show the plot
plt.show()


In [ ]:
# Actual VS Predicted Values

# y_test contains the actual target values for the test dataset
# y_pred contains the predicted values for the test dataset

# Create a scatter plot to visualize the relationship
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_linear, alpha=0.5)  # Plot actual vs. predicted values

# Add labels and title
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs. Predicted Values, Linear Model")

# Add a diagonal line for reference (perfect predictions)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], linestyle='--', color='red', lw=2)

# Show the plot
plt.show()